### Configuración inicial

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
import os

# Configuración del Master y Delta
master_url = "spark://spark-master:7077"

builder = SparkSession.builder \
    .appName("Ingesta_Bronze_SECOP") \
    .master(master_url) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "4") # Ajustado para entorno local

# Inicializar Spark con soporte para Delta Lake
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada con éxito")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-913268bc-5cb6-4bb7-91b9-e2d3da403529;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 189ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

SparkSession iniciada con éxito


### Lectura del archivo CSV 

In [ ]:
csv_path = "/app/data/SECOP_II_Contratos_Electronicos_20260126.csv"

print("Leyendo CSV ..")

df_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .load(csv_path)

print(f"Total de registros leídos: {df_raw.count()}")
df_raw.limit(5).toPandas()

Leyendo CSV ..


26/01/28 21:27:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 1:>                                                          (0 + 1) / 1]

### Limpiar nombres de columnas

In [ ]:
df_bronze = df_raw

for col_name in df_bronze.columns:
    clean_name = col_name.lower() \
        .replace(" ", "_") \
        .replace(".", "") \
        .replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u") \
        .replace("(", "").replace(")", "") \
        .replace(",", "").replace(";", "") # Añadimos estos para evitar el error de Delta
    
    df_bronze = df_bronze.withColumnRenamed(col_name, clean_name)

# 2. AGREGAR METADATA DE AUDITORÍA
df_bronze = df_bronze.withColumn(
    "_ingestion_time", 
    F.current_timestamp()
).withColumn(
    "_source_file", 
    F.input_file_name()
)

print("Columnas normalizadas y datos de auditoría agregados.")
print(f"Nuevas columnas: {df_bronze.columns[:5]}...") 
df_bronze.select("nombre_entidad", "_ingestion_time", "_source_file").show(5, False)

### Guardar datos en Delta

In [ ]:
# Usamos una ruta absoluta desde la raíz del contenedor
output_path = "/app/data/lakehouse/bronze/secop"

print(f"Guardando en capa Bronce en la ruta: {output_path}...")

df_bronze.repartition(10).write.format("delta") \
    .mode("overwrite") \
    .save(output_path)

print(f" ¡Ingesta completada! Datos protegidos en Delta Lake.")
print(f" Total columnas en Bronce: {len(df_bronze.columns)}")

### Leer Delta

In [ ]:
bronze_path = "/app/data/lakehouse/bronze/secop"

# Leemos usando el formato "delta"
df_bronze_check = spark.read.format("delta").load(bronze_path)

print(f" Lectura exitosa. Registros en el Lakehouse: {df_bronze_check.count()}")
# Ver las primeras 5 filas con Pandas 
df_bronze_check.limit(5).toPandas()